# Segmento 2: Function Calling

Diamo all'LLM un **tool** per cercare ricette. Sarà lui a decidere quando usarlo.

Il contesto della conversazione viene mantenuto. Vediamo se "dammene un'altra" ora funziona.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
import json, os

load_dotenv()
client = OpenAI()
qdrant = QdrantClient(host="localhost", port=6333)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-5.4-nano"
COLLECTION = "recipes"

# Verifichiamo che Qdrant abbia i dati della sessione 2
if not qdrant.collection_exists(COLLECTION):
    print(f"La collection '{COLLECTION}' non esiste!")
    print("Vai nel notebook session2/segment_2.ipynb ed eseguilo per crearla.")
else:
    info = qdrant.get_collection(COLLECTION)
    print(f"Collection '{COLLECTION}': {info.points_count} ricette")
    print(f"Modello chat: {CHAT_MODEL}")

Collection 'recipes': 1000 ricette
Modello chat: gpt-5.4-nano


## Definiamo il tool per l'LLM

Lo schema dice all'LLM: *"hai a disposizione una funzione `retrieve_recipes` che accetta una query e restituisce ricette"*.

L'LLM non esegue la funzione, ci dice **cosa vuole chiamare** e con quali argomenti.

In [2]:
# Lo schema del tool — è quello che l'LLM "vede"
tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_recipes",
            "description": "Cerca ricette nel database per similarità semantica. Usa questo tool quando l'utente chiede ricette o suggerimenti culinari.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "La query di ricerca per trovare ricette rilevanti"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# La funzione vera, quella che eseguiamo noi
def retrieve_recipes(query, top_n=5):
    """Cerca le ricette più rilevanti in Qdrant."""
    response = client.embeddings.create(input=[query], model=EMBED_MODEL)
    query_vector = response.data[0].embedding

    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        limit=top_n,
        with_payload=True,
    )

    recipes = []
    for point in results.points:
        recipes.append({
            "title": point.payload["title"],
            "ingredients": point.payload["ingredients"],
            "instructions": point.payload["instructions"],
            "score": round(point.score, 4),
        })
    return recipes

print("Tool definito: retrieve_recipes")
print(f"Schema:\n{json.dumps(tools[0]['function'], indent=2)}")

Tool definito: retrieve_recipes
Schema:
{
  "name": "retrieve_recipes",
  "description": "Cerca ricette nel database per similarit\u00e0 semantica. Usa questo tool quando l'utente chiede ricette o suggerimenti culinari.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "La query di ricerca per trovare ricette rilevanti"
      }
    },
    "required": [
      "query"
    ]
  }
}


In [3]:
# System prompt per l'agente
SYSTEM_PROMPT = """Sei un assistente culinario.
Aiuti gli utenti a trovare ricette dal nostro database.
Usa il tool retrieve_recipes quando l'utente cerca ricette o suggerimenti culinari.
"""

# Inizializziamo la conversazione
messages = [
    {"role": "system", "content": SYSTEM_PROMPT}
]
print("Conversazione inizializzata")

Conversazione inizializzata


## Prima domanda

Mandiamo la stessa query del notebook 1, ma questa volta l'LLM ha un tool a disposizione.

In [4]:
# Turno 1: l'utente chiede una ricetta
messages.append(
    {
        "role": "user",
        "content": "Cerco una ricetta con il pesce"
    }
)

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    tools=tools,
)

assistant_msg = response.choices[0].message

# L'LLM non ha risposto con testo: ha chiesto di chiamare un tool!
print("=== Tool Call — JSON raw ===\n")
for tc in assistant_msg.tool_calls:
    print(json.dumps({
        "id": tc.id,
        "function": {
            "name": tc.function.name,
            "arguments": tc.function.arguments
        }
    }, indent=2))

print("\n=== Versione leggibile ===\n")
for tc in assistant_msg.tool_calls:
    args = json.loads(tc.function.arguments)
    print(f"L'LLM vuole chiamare: {tc.function.name}()")
    for k, v in args.items():
        print(f"  {k}: {v}")

=== Tool Call — JSON raw ===

{
  "id": "call_FAMqELghVIzgVvDUjXlXULA2",
  "function": {
    "name": "retrieve_recipes",
    "arguments": "{\"query\":\"ricetta con il pesce\"}"
  }
}

=== Versione leggibile ===

L'LLM vuole chiamare: retrieve_recipes()
  query: ricetta con il pesce


## Eseguiamo il tool e restituiamo il risultato

L'LLM ci ha detto **cosa** vuole fare. Ora tocca a noi **eseguire** la funzione e mandargli indietro i risultati.

In [5]:
# Eseguiamo il tool call
tool_call = assistant_msg.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = retrieve_recipes(**args)

print("Risultati della ricerca:\n")
for r in result:
    print(f"  {r['score']:.4f}  {r['title']}")

# Aggiungiamo alla conversazione: il messaggio dell'assistente + il risultato del tool
messages.append(assistant_msg)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result)
})

# Ora l'LLM genera la risposta finale con i dati reali
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    tools=tools,
)

final_msg = response.choices[0].message
messages.append(final_msg)

print(f"\n--- Risposta dell'assistente ---\n\n{final_msg.content}")

Risultati della ricerca:

  0.4823  Whole Branzino Roasted in Salt
  0.4739  Crispy Fish with Brown Butter Sauce and Kohlrabi Salad
  0.4699  Crispy Za'atar Fish With Israeli Couscous, Swiss Chard, and Feta
  0.4655  Black Sea Bass with Moroccan Vegetables and Chile Sauce
  0.4640  Monkfish with Ratatouille



--- Risposta dell'assistente ---

Certo! Nel nostro database ho trovato queste ricette a base di pesce: dimmi quale stile ti piace (al forno, impanata/croccante, sughi, ecc.) e la preparo passo‑passo.

1) **Whole Branzino Roasted in Salt** (branzino al forno in crosta di sale)  
2) **Crispy Fish with Brown Butter Sauce and Kohlrabi Salad** (pesce croccante con burro nocciola e insalata)  
3) **Crispy Za’atar Fish With Israeli Couscous, Swiss Chard, and Feta** (pesce speziato za’atar con couscous e feta)  
4) **Black Sea Bass with Moroccan Vegetables and Chile Sauce** (orata/sea bass con verdure marocchine e salsa)  
5) **Monkfish with Ratatouille** (monkfish con ratatouille)

Vuoi **pesce intero o filetti**? E preferisci **leggero** o **più saporito/cremoso**?


## Il momento della verità: "dammene un'altra"

Stessa richiesta che nel notebook 1 dava risultati senza senso.

Questa volta l'LLM ha il **contesto della conversazione**, sa che stavamo parlando di ricette con il pesce.

In [6]:
# Turno 2: la richiesta che prima falliva
messages.append({"role": "user", "content": "Dammene un'altra"})

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    tools=tools,
)

assistant_msg_2 = response.choices[0].message

# Vediamo cosa ha fatto l'LLM
if assistant_msg_2.tool_calls:
    print("L'LLM ha fatto un nuovo tool call!\n")
    for tc in assistant_msg_2.tool_calls:
        args = json.loads(tc.function.arguments)
        print(f"  Funzione: {tc.function.name}")
        print(f"  Argomenti: {args}")
        print(f"\n  Ha capito dal contesto che 'un'altra' si riferisce a ricette con il pesce!")
else:
    print("L'LLM ha risposto direttamente (aveva già risultati dal turno precedente):\n")
    print(assistant_msg_2.content)

L'LLM ha fatto un nuovo tool call!

  Funzione: retrieve_recipes
  Argomenti: {'query': 'ricetta pesce'}

  Ha capito dal contesto che 'un'altra' si riferisce a ricette con il pesce!


In [7]:
# Completiamo il turno 2
if assistant_msg_2.tool_calls:
    tool_call_2 = assistant_msg_2.tool_calls[0]
    args_2 = json.loads(tool_call_2.function.arguments)
    result_2 = retrieve_recipes(**args_2)

    print("Risultati:\n")
    for r in result_2:
        print(f"  {r['score']:.4f}  {r['title']}")

    messages.append(assistant_msg_2)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call_2.id,
        "content": json.dumps(result_2)
    })

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        tools=tools,
    )
    final_msg_2 = response.choices[0].message
    messages.append(final_msg_2)
    print(f"\n--- Risposta dell'assistente ---\n\n{final_msg_2.content}")
else:
    messages.append(assistant_msg_2)
    print("(Risposta già mostrata sopra)")

Risultati:

  0.4960  Whole Branzino Roasted in Salt
  0.4851  Crispy Za'atar Fish With Israeli Couscous, Swiss Chard, and Feta
  0.4815  Crispy Fish with Brown Butter Sauce and Kohlrabi Salad
  0.4772  Black Sea Bass with Moroccan Vegetables and Chile Sauce
  0.4743  Roasted Black Bass with Orange-Flower Water



--- Risposta dell'assistente ---

Ecco **un’altra** ricetta di pesce dal nostro database:

**Roasted Black Bass with Orange-Flower Water** (branzino/occhiata nera al forno con acqua di fiori d’arancio)

Se mi dici **che pesce hai** (branzino, salmone, merluzzo, orata, ecc.) e se lo vuoi **in forno o in padella**, te la adatto e ti scrivo anche gli step completi.


## Cosa abbiamo visto

Con il function calling:

1. L'LLM **decide da solo** quando cercare nel database
2. Il **contesto della conversazione** viene mantenuto, "dammene un'altra" funziona
3. L'LLM **non esegue** codice, ci dice cosa vuole fare e noi eseguiamo

Prossimo passo: aggiungiamo un secondo tool per fare qualcosa **nel mondo reale**, torniamo alle slides.